# LeetCode 1874 — Minimize Product Sum of Two Arrays bằng Python

Link đề: https://leetcode.com/problems/minimize-product-sum-of-two-arrays/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

> **Bài Premium.** Đề bài trong notebook được viết lại theo hiểu biết về bài (nguồn: bản mô tả của doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode:

```python
class Solution:
    def minProductSum(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

Lộ trình:
1. **Thử mọi hoán vị** của `nums1` (`O(n! · n)`) — đáp án chuẩn cho `n ≤ 8`.
2. **DP trên bitmask** (bài toán phân công) — `O(n · 2ⁿ)`, vẫn vét cạn nhưng nhanh hơn nhiều.
3. **Tham lam: sắp `nums1` tăng, `nums2` giảm, ghép theo vị trí** — `O(n log n)`; chứng minh bằng **bất đẳng thức sắp xếp lại** (rearrangement inequality) và lập luận đổi chỗ.
4. **Counting sort** vì giá trị `≤ 100` — ghép hai bảng đếm bằng hai con trỏ, `O(n + V)`.
5. **Heap** (min-heap × max-heap) — cùng thứ tự ghép, minh họa cấu trúc dữ liệu khác.

## 1. Mục tiêu học

1. Hiểu "được sắp xếp lại `nums1` tùy ý" nghĩa là tìm **một cách ghép** (song ánh) giữa phần tử của `nums1` và `nums2` để tổng tích nhỏ nhất.
2. Phát biểu và chứng minh **bất đẳng thức sắp xếp lại**: ghép **lớn với nhỏ** cho tổng tích nhỏ nhất; ghép **lớn với lớn** cho tổng tích lớn nhất.
3. Dùng **lập luận đổi chỗ (exchange argument)** để chứng minh tham lam — kỹ thuật dùng lại cho rất nhiều bài sắp xếp.
4. Tận dụng miền giá trị nhỏ (`≤ 100`) để thay sắp xếp `O(n log n)` bằng **đếm** `O(n + V)`.
5. Kiểm chứng tham lam bằng vét cạn hoán vị và DP bitmask trên input ngẫu nhiên.

## 2. Đề bài & ví dụ

**Tổng tích (product sum)** của hai mảng cùng độ dài `a`, `b` là `Σ a[i] · b[i]` với mọi `0 ≤ i < n`.
Ví dụ `a = [1,2,3,4]`, `b = [5,2,3,1]` ⇒ `1·5 + 2·2 + 3·3 + 4·1 = 22`.

Cho hai mảng `nums1`, `nums2` độ dài `n`. Trả về tổng tích **nhỏ nhất** nếu được phép **sắp xếp lại thứ tự các phần tử của `nums1`** (giữ nguyên `nums2`).

**Ví dụ 1:** `nums1 = [5,3,4,2]`, `nums2 = [4,2,2,5]` → `40`. Sắp `nums1` thành `[3,5,4,2]`: `3·4 + 5·2 + 4·2 + 2·5 = 40`.

**Ví dụ 2:** `nums1 = [2,1,4,5,7]`, `nums2 = [3,2,4,8,6]` → `65`. Sắp `nums1` thành `[5,7,4,1,2]`: `5·3 + 7·2 + 4·4 + 1·8 + 2·6 = 65`.

**Ràng buộc:**
- `n == nums1.length == nums2.length`, `1 ≤ n ≤ 10⁵`
- `1 ≤ nums1[i], nums2[i] ≤ 100`

In [1]:
import random
import time
import heapq
from itertools import permutations
from functools import lru_cache
from typing import List

examples = [
    (([5, 3, 4, 2], [4, 2, 2, 5]), 40),
    (([2, 1, 4, 5, 7], [3, 2, 4, 8, 6]), 65),
]
def product_sum(a, b):
    return sum(x * y for x, y in zip(a, b))

for (a, b), want in examples:
    print(f"nums1={a}, nums2={b} -> kỳ vọng {want}")
print("kiểm tra sắp xếp trong đề:", product_sum([3, 5, 4, 2], [4, 2, 2, 5]), product_sum([5, 7, 4, 1, 2], [3, 2, 4, 8, 6]))

nums1=[5, 3, 4, 2], nums2=[4, 2, 2, 5] -> kỳ vọng 40
nums1=[2, 1, 4, 5, 7], nums2=[3, 2, 4, 8, 6] -> kỳ vọng 65
kiểm tra sắp xếp trong đề: 40 65


## 3. Kiến thức nền

### 3.1 `zip` và tổng tích

`zip(a, b)` ghép từng cặp cùng vị trí; `sum(x * y for x, y in zip(a, b))` là tổng tích.

In [2]:
print(list(zip([1, 2, 3], [4, 5, 6])), product_sum([1, 2, 3], [4, 5, 6]))

[(1, 4), (2, 5), (3, 6)] 32


### 3.2 Sắp xếp tăng / giảm

`sorted(a)` tăng dần, `sorted(a, reverse=True)` giảm dần. `list.sort()` sắp tại chỗ (sửa input — cẩn thận!).

In [3]:
a = [5, 3, 4, 2]
print(sorted(a), sorted(a, reverse=True), a, "(a không đổi)")

[2, 3, 4, 5] [5, 4, 3, 2] [5, 3, 4, 2] (a không đổi)


### 3.3 "Sắp lại `nums1`" = "chọn cách ghép"

Vì chỉ `nums1` được hoán vị còn `nums2` cố định, mọi cách sắp tương ứng với một **hoán vị** `π`: phần tử `nums1[π(i)]` đứng cạnh `nums2[i]`. Tổng tích chỉ phụ thuộc **tập các cặp** được ghép, không phụ thuộc thứ tự liệt kê các cặp. Vì vậy ta cũng có thể sắp `nums2` cho tiện tính toán — tổng không đổi.

In [4]:
a, b = [3, 5, 4, 2], [4, 2, 2, 5]
pairs = sorted(zip(a, b), key=lambda p: p[1])   # đổi thứ tự liệt kê các cặp
print(pairs, "tổng =", sum(x * y for x, y in pairs), "(không đổi)")

[(5, 2), (4, 2), (3, 4), (2, 5)] tổng = 40 (không đổi)


### 3.4 Bảng đếm cho giá trị nhỏ

Giá trị trong `[1, 100]` ⇒ `cnt = [0] * 101`, `cnt[v] += 1`. Duyệt `v` tăng dần là đã "sắp xếp" mảng trong `O(n + V)`.

In [5]:
cnt = [0] * 11
for v in [5, 3, 4, 2, 3]:
    cnt[v] += 1
print([(v, c) for v, c in enumerate(cnt) if c])

[(2, 1), (3, 2), (4, 1), (5, 1)]


## 4. Ý tưởng nền tảng / trực giác

Bài toán nhỏ nhất: hai số `x ≤ y` trong `nums1` ghép với hai số `p ≤ q` trong `nums2`. Hai cách:
- **cùng chiều**: `x·p + y·q`
- **ngược chiều**: `x·q + y·p`

Hiệu: `(x·p + y·q) − (x·q + y·p) = (y − x)(q − p) ≥ 0`. Vậy **ghép ngược chiều luôn không tệ hơn**: số lớn của bên này nên gặp số nhỏ của bên kia.

Áp dụng cho ví dụ 1: sắp `nums1 = [2,3,4,5]` tăng, `nums2 = [5,4,2,2]` giảm ⇒ `2·5 + 3·4 + 4·2 + 5·2 = 10 + 12 + 8 + 10 = 40`. ✓

In [6]:
x, y, p, q = 2, 5, 3, 7
print("cùng chiều:", x * p + y * q, "| ngược chiều:", x * q + y * p, "| hiệu = (y-x)(q-p) =", (y - x) * (q - p))
a, b = [5, 3, 4, 2], [4, 2, 2, 5]
print("ghép tăng × giảm:", list(zip(sorted(a), sorted(b, reverse=True))), "->", product_sum(sorted(a), sorted(b, reverse=True)))

cùng chiều: 41 | ngược chiều: 29 | hiệu = (y-x)(q-p) = 12
ghép tăng × giảm: [(2, 5), (3, 4), (4, 2), (5, 2)] -> 40


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

**Ý tưởng:** duyệt mọi hoán vị của `nums1`, tính tổng tích, lấy min. Đúng theo định nghĩa.

In [7]:
def min_product_perm(nums1, nums2):
    return min(product_sum(p, nums2) for p in permutations(nums1))

for (a, b), want in examples:
    print(min_product_perm(a, b), "kỳ vọng", want)

40 kỳ vọng 40
65 kỳ vọng 65


**Độ phức tạp:** `n!` hoán vị × `O(n)` ⇒ `O(n! · n)`. `n = 10` đã là ~3,6 triệu hoán vị. Space `O(n)`.

**Nút thắt:** nhiều hoán vị có chung "phần đầu" — tính lại tổng từ đầu. DP bitmask gom các hoán vị có cùng **tập phần tử đã dùng**.

In [8]:
random.seed(0)
for n in [6, 7, 8]:
    a = [random.randint(1, 100) for _ in range(n)]; b = [random.randint(1, 100) for _ in range(n)]
    t0 = time.perf_counter(); min_product_perm(a, b)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=6: 0.000s
n=7: 0.003s
n=8: 0.023s


### 5.1 Cách 1b — DP bitmask (bài toán phân công)

`f(mask)` = tổng nhỏ nhất khi đã ghép `nums2[0..k−1]` (với `k = popcount(mask)`) với tập phần tử `nums1` trong `mask`. Chuyển: chọn phần tử `i ∉ mask` ghép với `nums2[k]`.

`f(mask | 1<<i) = min(f(mask) + nums1[i] · nums2[k])`. Đáp án `f(full)`.

In [9]:
def min_product_bitmask(nums1, nums2):
    n = len(nums1)
    INF = float("inf")
    f = [INF] * (1 << n)
    f[0] = 0
    for mask in range(1 << n):
        if f[mask] == INF:
            continue
        k = bin(mask).count("1")
        if k == n:
            continue
        for i in range(n):
            if not mask >> i & 1:
                nm = mask | 1 << i
                v = f[mask] + nums1[i] * nums2[k]
                if v < f[nm]:
                    f[nm] = v
    return f[-1]

for (a, b), want in examples:
    print(min_product_bitmask(a, b), "kỳ vọng", want)

40 kỳ vọng 40
65 kỳ vọng 65


**Độ phức tạp:** `O(n · 2ⁿ)` thời gian, `O(2ⁿ)` bộ nhớ — `n = 16` khoảng 10⁶ phép. Vẫn mũ, nhưng là công cụ kiểm chứng tốt cho `n` lớn hơn so với hoán vị.

## 6. Ý tưởng thuật toán / bước đột phá

**Bất đẳng thức sắp xếp lại.** Với `a₁ ≤ a₂ ≤ … ≤ aₙ` và `b₁ ≥ b₂ ≥ … ≥ bₙ`, mọi hoán vị `σ`:

`Σ aᵢ · bᵢ ≤ Σ aᵢ · b_{σ(i)}`.

**Lập luận đổi chỗ:** nếu trong một cách ghép có hai cặp `(x, q)` và `(y, p)` với `x < y` và `p < q` — tức số nhỏ ghép số lớn **và** số lớn ghép số nhỏ, đó là ghép "ngược chiều" — tốt. Nếu lại có `(x, p)` và `(y, q)` (cùng chiều) thì đổi thành `(x, q), (y, p)` giảm tổng `(y − x)(q − p) ≥ 0`. Lặp lại tới khi mọi cặp đều ngược chiều — đó chính là ghép "tăng × giảm".

**Quy tắc:** sắp `nums1` tăng, `nums2` giảm, nhân theo vị trí.

**Invariant cho counting sort:** duyệt `nums1` theo giá trị tăng (con trỏ `i` từ 1 lên 100) và `nums2` theo giá trị giảm (con trỏ `j` từ 100 xuống 1); mỗi bước ghép `min(cnt1[i], cnt2[j])` cặp `(i, j)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp tăng × giảm

In [10]:
def min_product_sort(nums1, nums2):
    a = sorted(nums1)
    b = sorted(nums2, reverse=True)
    return sum(x * y for x, y in zip(a, b))

for (a, b), want in examples:
    print(min_product_sort(a, b), "kỳ vọng", want)

40 kỳ vọng 40
65 kỳ vọng 65


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def minProductSum(self, nums1: List[int], nums2: List[int]) -> int:
        a = sorted(nums1)
        b = sorted(nums2, reverse=True)
        return sum(x * y for x, y in zip(a, b))
```

**Độ phức tạp:** Time `O(n log n)`, Space `O(n)` (hai mảng đã sắp).

### 7.2 Cách 3 — Đếm (counting sort) + hai con trỏ

Từng đoạn:
1. `c1[v]`, `c2[v]` — số lần giá trị `v` xuất hiện trong `nums1`, `nums2`.
2. `i = 1` (giá trị nhỏ nhất của `nums1`), `j = 100` (giá trị lớn nhất của `nums2`).
3. Bỏ qua `c1[i] == 0` (tăng `i`), `c2[j] == 0` (giảm `j`).
4. `t = min(c1[i], c2[j])`: ghép `t` cặp `(i, j)`, cộng `t · i · j`, trừ `t` khỏi cả hai.
5. Dừng khi đã ghép đủ `n` cặp.

In [11]:
def min_product_count(nums1, nums2, V=100):
    c1 = [0] * (V + 1)
    c2 = [0] * (V + 1)
    for x in nums1:
        c1[x] += 1
    for y in nums2:
        c2[y] += 1
    i, j = 1, V
    total = 0
    remaining = len(nums1)
    while remaining:
        while c1[i] == 0:
            i += 1
        while c2[j] == 0:
            j -= 1
        t = min(c1[i], c2[j])
        total += t * i * j
        c1[i] -= t
        c2[j] -= t
        remaining -= t
    return total

for (a, b), want in examples:
    print(min_product_count(a, b), "kỳ vọng", want)

40 kỳ vọng 40
65 kỳ vọng 65


**Độ phức tạp:** đếm `O(n)`, hai con trỏ mỗi con đi tối đa `V` bước và mỗi vòng `while remaining` hoặc làm hết một bảng đếm ⇒ `O(n + V)` với `V = 100`. Space `O(V)`.

Code nộp LeetCode của Cách 3 (dùng ở mục 9 — nhanh nhất cho `n = 10⁵`).

### 7.3 Cách 4 — Hai heap

Đưa `nums1` vào min-heap, `nums2` vào max-heap (đổi dấu); lần lượt pop một phần tử từ mỗi heap và nhân. Cùng thứ tự ghép như Cách 2 — hữu ích khi dữ liệu đến dạng **luồng** và chỉ cần `k` cặp đầu tiên.

In [12]:
def min_product_heap(nums1, nums2):
    h1 = list(nums1)
    h2 = [-y for y in nums2]
    heapq.heapify(h1)
    heapq.heapify(h2)
    total = 0
    while h1:
        total += heapq.heappop(h1) * -heapq.heappop(h2)
    return total

for (a, b), want in examples:
    print(min_product_heap(a, b), "kỳ vọng", want)

40 kỳ vọng 40
65 kỳ vọng 65


**Độ phức tạp:** heapify `O(n)` + `n` lần pop `O(log n)` ⇒ Time `O(n log n)`, Space `O(n)`.

### 7.4 Khôi phục chính hoán vị của `nums1`

Đề chỉ hỏi giá trị nhỏ nhất, nhưng để **giải thích** (như phần Explanation của đề) ta có thể dựng cách sắp `nums1` cụ thể: sắp **chỉ số** của `nums2` theo giá trị giảm, rồi đặt phần tử nhỏ nhất của `nums1` vào chỉ số có `nums2` lớn nhất, v.v.

In [13]:
def best_arrangement(nums1, nums2):
    order = sorted(range(len(nums2)), key=lambda i: -nums2[i])    # chỉ số nums2 theo giá trị giảm
    res = [0] * len(nums1)
    for idx, x in zip(order, sorted(nums1)):
        res[idx] = x
    return res

for (a, b), want in examples:
    arr = best_arrangement(a, b)
    print(f"nums1 sắp lại = {arr}, tổng tích = {product_sum(arr, b)} (kỳ vọng {want})")

nums1 sắp lại = [3, 4, 5, 2], tổng tích = 40 (kỳ vọng 40)
nums1 sắp lại = [5, 7, 4, 1, 2], tổng tích = 65 (kỳ vọng 65)


Cách sắp có thể khác với ví dụ của đề (khi có giá trị bằng nhau), nhưng tổng tích luôn bằng giá trị tối ưu.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 `nums1 = [2,1,4,5,7]`, `nums2 = [3,2,4,8,6]`:

| bước | a (tăng) | b (giảm) | tích | tổng |
|---|---|---|---|---|
| 1 | 1 | 8 | 8 | 8 |
| 2 | 2 | 6 | 12 | 20 |
| 3 | 4 | 4 | 16 | 36 |
| 4 | 5 | 3 | 15 | 51 |
| 5 | 7 | 2 | 14 | **65** |

In [14]:
def trace_sort(nums1, nums2):
    a, b = sorted(nums1), sorted(nums2, reverse=True); total = 0
    print("a tăng:", a, "| b giảm:", b)
    for k, (x, y) in enumerate(zip(a, b), 1):
        total += x * y
        print(f"bước {k}: {x} × {y} = {x * y:3d}, tổng = {total}")
    return total

trace_sort([2, 1, 4, 5, 7], [3, 2, 4, 8, 6])

a tăng: [1, 2, 4, 5, 7] | b giảm: [8, 6, 4, 3, 2]
bước 1: 1 × 8 =   8, tổng = 8
bước 2: 2 × 6 =  12, tổng = 20
bước 3: 4 × 4 =  16, tổng = 36
bước 4: 5 × 3 =  15, tổng = 51
bước 5: 7 × 2 =  14, tổng = 65


65

In [15]:
def trace_count(nums1, nums2, V=10):
    c1 = [0] * (V + 1); c2 = [0] * (V + 1)
    for x in nums1: c1[x] += 1
    for y in nums2: c2[y] += 1
    print("c1:", {v: c for v, c in enumerate(c1) if c}, "| c2:", {v: c for v, c in enumerate(c2) if c})
    i, j, total, rem = 1, V, 0, len(nums1)
    while rem:
        while c1[i] == 0: i += 1
        while c2[j] == 0: j -= 1
        t = min(c1[i], c2[j]); total += t * i * j
        print(f"ghép {t} cặp ({i}, {j}) -> +{t * i * j}, tổng = {total}")
        c1[i] -= t; c2[j] -= t; rem -= t
    return total

trace_count([5, 3, 4, 2], [4, 2, 2, 5])

c1: {2: 1, 3: 1, 4: 1, 5: 1} | c2: {2: 2, 4: 1, 5: 1}
ghép 1 cặp (2, 5) -> +10, tổng = 10
ghép 1 cặp (3, 4) -> +12, tổng = 22
ghép 1 cặp (4, 2) -> +8, tổng = 30
ghép 1 cặp (5, 2) -> +10, tổng = 40


40

Ở ví dụ 1 bảng đếm `nums2` có `2` xuất hiện hai lần: con trỏ `j` dừng ở giá trị `2` và ghép **một lúc** với các giá trị lớn nhất còn lại của `nums1` (lần lượt `4` rồi `5`, vì `c1[4] = 1`).

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minProductSum(self, nums1: List[int], nums2: List[int]) -> int:
        c1 = [0] * 101
        c2 = [0] * 101
        for x in nums1:
            c1[x] += 1
        for y in nums2:
            c2[y] += 1
        i, j = 1, 100
        total = 0
        remaining = len(nums1)
        while remaining:
            while c1[i] == 0:
                i += 1
            while c2[j] == 0:
                j -= 1
            t = c1[i] if c1[i] < c2[j] else c2[j]
            total += t * i * j
            c1[i] -= t
            c2[j] -= t
            remaining -= t
        return total

sol = Solution()
for (a, b), want in examples:
    got = sol.minProductSum(a, b)
    print(got, "OK" if got == want else "SAI")

40 OK
65 OK


## 10. Vì sao thuật toán đúng

**Bổ đề (đổi chỗ hai cặp).** Với `x ≤ y`, `p ≤ q`: `x·q + y·p ≤ x·p + y·q` vì hiệu là `(y − x)(q − p) ≥ 0`.

**Định lý.** Sắp `a` tăng, `b` giảm. Ghép `aᵢ` với `bᵢ` cho tổng nhỏ nhất.

*Chứng minh.* Xét một cách ghép tối ưu bất kỳ: `aᵢ` ghép với `b_{σ(i)}`. Ta muốn phần tử `a` nhỏ hơn nhận phần tử `b` **không nhỏ hơn**. Giả sử có `i < j` (nên `aᵢ ≤ aⱼ`) mà `b_{σ(i)} < b_{σ(j)}` — một cặp "cùng chiều". Theo bổ đề với `x = aᵢ, y = aⱼ, p = b_{σ(i)}, q = b_{σ(j)}`, đổi hai phần tử `b` cho nhau không làm tổng tăng. Mỗi lần đổi giảm số "nghịch thế" của `σ` theo thứ tự `b`, nên sau hữu hạn bước ta đạt cách ghép mà `b_{σ(i)}` không tăng theo `i` — tức đúng ghép "tăng × giảm" (khi có giá trị bằng nhau, các cách ghép khác nhau cho cùng tổng). ∎

**Cách 3** liệt kê đúng dãy cặp của Cách 2 theo khối giá trị: con trỏ `i` đi qua `nums1` theo thứ tự tăng, `j` đi qua `nums2` theo thứ tự giảm, mỗi khối `t` cặp giống nhau ⇒ cùng tổng.

**Cách 4** pop min của `nums1` và max của `nums2` mỗi bước — cũng đúng dãy cặp đó.

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁵`, `V = 100` là giá trị lớn nhất.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hoán vị | `O(n! · n)` | `O(n)` | `n ≤ 9` |
| 1b. DP bitmask | `O(n · 2ⁿ)` | `O(2ⁿ)` | `n ≤ 18` |
| 2. Sort tăng × giảm | `O(n log n)` | `O(n)` | đơn giản nhất |
| 3. Counting + 2 con trỏ | `O(n + V)` | `O(V)` | **nộp bài** |
| 4. Hai heap | `O(n log n)` | `O(n)` | dạng luồng / top-k |

Độ lớn kết quả: tối đa `10⁵ · 100 · 100 = 10⁹` — vừa `int32` có dấu (2.1·10⁹) nhưng nên dùng `long` trong C++/Java cho an toàn.

In [17]:
random.seed(1)
n = 10**5
a = [random.randint(1, 100) for _ in range(n)]; b = [random.randint(1, 100) for _ in range(n)]
for name, fn in [("sort", min_product_sort), ("count", min_product_count), ("heap", min_product_heap), ("submit", Solution().minProductSum)]:
    t0 = time.perf_counter(); r = fn(a, b)
    print(f"{name:6s}: {r}  {time.perf_counter() - t0:.4f}s")
print("tổng lớn nhất có thể:", 10**5 * 100 * 100, "| 2^31 - 1 =", 2**31 - 1)

sort  : 171273764  0.0279s


count : 171273764  0.0059s


heap  : 171273764  0.0778s


submit: 171273764  0.0057s
tổng lớn nhất có thể: 1000000000 | 2^31 - 1 = 2147483647


## 12. Bộ test

Đối chiếu với hoán vị (`n ≤ 7`) và DP bitmask (`n ≤ 12`); edge case giá trị bằng nhau, `n = 1`, mảng đã "ngược chiều" sẵn.

In [18]:
fns = [min_product_bitmask, min_product_sort, min_product_count, min_product_heap,
       lambda a, b: Solution().minProductSum(a, b)]

for (a, b), want in examples:
    assert min_product_perm(a, b) == want
    for fn in fns:
        assert fn(a, b) == want

edge = [
    (([7], [9]), 63),
    (([1, 1, 1], [5, 6, 7]), 18),
    (([1, 100], [1, 100]), 200),            # 1*100 + 100*1
    (([100] * 4, [100] * 4), 40000),
    (([1, 2, 3], [3, 2, 1]), 10),           # sắp cùng chiều sẵn: 1*3+2*2+3*1 = 10 đã tối ưu
]
for (a, b), want in edge:
    assert min_product_perm(a, b) == want, (a, b)
    for fn in fns:
        assert fn(a, b) == want, (fn, a, b)

random.seed(2024)
for _ in range(400):
    n = random.randint(1, 7)
    hi = random.choice([3, 10, 100])
    a = [random.randint(1, hi) for _ in range(n)]; b = [random.randint(1, hi) for _ in range(n)]
    want = min_product_perm(a, b)
    for fn in fns:
        assert fn(a, b) == want, (fn, a, b)
print("ví dụ + edge + 400 bộ ngẫu nhiên khớp vét cạn hoán vị")

ví dụ + edge + 400 bộ ngẫu nhiên khớp vét cạn hoán vị


In [19]:
random.seed(7)
for _ in range(60):
    n = random.randint(8, 12)
    a = [random.randint(1, 100) for _ in range(n)]; b = [random.randint(1, 100) for _ in range(n)]
    want = min_product_bitmask(a, b)
    assert min_product_sort(a, b) == min_product_count(a, b) == min_product_heap(a, b) == want
print("n = 8..12: tham lam khớp DP bitmask")

# input không bị sửa
a = [5, 3, 4, 2]; b = [4, 2, 2, 5]
Solution().minProductSum(a, b); min_product_sort(a, b)
assert a == [5, 3, 4, 2] and b == [4, 2, 2, 5]
print("input không bị thay đổi")

n = 8..12: tham lam khớp DP bitmask
input không bị thay đổi


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Sắp cả hai mảng **cùng chiều**

Đây là cách cho tổng **lớn nhất** (bất đẳng thức sắp xếp lại theo chiều ngược lại).

In [20]:
def wrong_same_order(nums1, nums2):
    return product_sum(sorted(nums1), sorted(nums2))

for (a, b), want in examples:
    print("sai =", wrong_same_order(a, b), "| đúng =", want)

sai = 51 | đúng = 40
sai = 110 | đúng = 65


**Sửa:** một mảng tăng, một mảng giảm.

### Lỗi 2 — Chỉ sắp `nums1` (giảm dần) mà giữ `nums2` nguyên thứ tự

Ghép `nums1` giảm dần với `nums2` **chưa sắp** không bảo đảm ngược chiều.

In [21]:
def wrong_sort_one(nums1, nums2):
    return product_sum(sorted(nums1, reverse=True), nums2)

for (a, b), want in examples:
    print("sai =", wrong_sort_one(a, b), "| đúng =", want)

sai = 44 | đúng = 40
sai = 69 | đúng = 65


**Sửa:** sắp cả hai (hoặc gán theo thứ tự chỉ số của `nums2` như `best_arrangement`).

### Lỗi 3 — Tham lam "mỗi `nums2[i]` chọn phần tử nhỏ nhất còn lại của `nums1`" theo thứ tự xuất hiện

Duyệt `nums2` từ trái sang phải và luôn lấy phần tử nhỏ nhất còn lại — bỏ qua việc phần tử nhỏ nên dành cho `nums2` **lớn**.

In [22]:
def wrong_left_to_right(nums1, nums2):
    pool = sorted(nums1); total = 0
    for y in nums2:
        total += pool.pop(0) * y
    return total

for (a, b), want in examples:
    print("sai =", wrong_left_to_right(a, b), "| đúng =", want)

sai = 47 | đúng = 40
sai = 105 | đúng = 65


**Sửa:** xử lý `nums2` theo thứ tự **giảm dần** (lớn nhất nhận nhỏ nhất).

### Lỗi 4 — Counting sort quên trừ số lượng ở **cả hai** bảng đếm

Chỉ trừ `c1[i]` ⇒ con trỏ `j` đứng yên mãi ở giá trị lớn nhất của `nums2` — mọi phần tử `nums1` đều nhân với max(nums2).

In [23]:
def wrong_count(nums1, nums2):
    c1 = [0] * 101; c2 = [0] * 101
    for x in nums1: c1[x] += 1
    for y in nums2: c2[y] += 1
    i, j, total, rem = 1, 100, 0, len(nums1)
    while rem:
        while c1[i] == 0: i += 1
        while c2[j] == 0: j -= 1
        t = min(c1[i], c2[j]); total += t * i * j
        c1[i] -= t; rem -= t                 # SAI: quên c2[j] -= t
    return total

for (a, b), want in examples:
    print("sai =", wrong_count(a, b), "| đúng =", want)

sai = 70 | đúng = 40
sai = 152 | đúng = 65


**Sửa:** `c1[i] −= t` **và** `c2[j] −= t`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Được sắp lại một/cả hai mảng, tối ưu tổng tích" ⇒ **bất đẳng thức sắp xếp lại**: min ⇔ ngược chiều, max ⇔ cùng chiều.
- Mọi bài "ghép cặp tối ưu với hàm tách được" nên thử **lập luận đổi chỗ hai cặp** trước khi nghĩ tới DP.
- Giá trị nhỏ (`≤ 100`, `≤ 1000`) ⇒ counting sort thay sort.
- Kiểm chứng tham lam bằng DP bitmask (`n ≤ 15`) — nhanh hơn liệt kê hoán vị nhiều.
- Họ bài: 1877 *Minimize Maximum Pair Sum* (ghép nhỏ–lớn trong **một** mảng), 870 *Advantage Shuffle*, 455 *Assign Cookies*.

## 15. Biến thể của bài

1. **Tổng tích lớn nhất:** sắp cùng chiều.
2. **Được hoán vị cả hai mảng:** như bài gốc (chỉ cách ghép quan trọng).
3. **Có số âm:** bất đẳng thức sắp xếp lại vẫn đúng với số thực bất kỳ — sắp tăng × giảm vẫn tối ưu.
4. **Chỉ chọn `k` cặp** (mỗi bên chọn `k` phần tử, tối thiểu tổng tích, số dương): chọn `k` nhỏ nhất của mỗi bên? Không đơn giản — xem Bài 5.
5. **Tối thiểu `max(aᵢ · bᵢ)`** thay vì tổng: cũng ghép ngược chiều (lập luận đổi chỗ tương tự).

In [24]:
def max_product_sum(a, b):
    return product_sum(sorted(a), sorted(b))

def min_product_with_negatives(a, b):
    return product_sum(sorted(a), sorted(b, reverse=True))

random.seed(3)
for _ in range(300):
    n = random.randint(1, 6)
    a = [random.randint(-10, 10) for _ in range(n)]; b = [random.randint(-10, 10) for _ in range(n)]
    assert min_product_with_negatives(a, b) == min(product_sum(p, b) for p in permutations(a))
    assert max_product_sum(a, b) == max(product_sum(p, b) for p in permutations(a))
print("biến thể có số âm + tổng lớn nhất: khớp vét cạn")

def min_max_product(a, b):
    return max(x * y for x, y in zip(sorted(a), sorted(b, reverse=True)))
for _ in range(300):
    n = random.randint(1, 6)
    a = [random.randint(1, 20) for _ in range(n)]; b = [random.randint(1, 20) for _ in range(n)]
    assert min_max_product(a, b) == min(max(x * y for x, y in zip(p, b)) for p in permutations(a))
print("biến thể min(max tích): ghép ngược chiều cũng tối ưu (số dương)")

biến thể có số âm + tổng lớn nhất: khớp vét cạn
biến thể min(max tích): ghép ngược chiều cũng tối ưu (số dương)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị | thử mọi cách sắp | `O(n!·n)` | `O(n)` | kiểm chứng tí hon |
| 1b. DP bitmask | phân công tối ưu | `O(n·2ⁿ)` | `O(2ⁿ)` | kiểm chứng vừa |
| 2. Sort tăng × giảm | rearrangement | `O(n log n)` | `O(n)` | mặc định |
| 3. Counting | hai bảng đếm | `O(n + V)` | `O(V)` | **nộp bài** khi giá trị nhỏ |
| 4. Hai heap | pop min × pop max | `O(n log n)` | `O(n)` | luồng dữ liệu |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `minProductSum([1, 2], [3, 4])` và `minProductSum([3, 3, 3], [1, 2, 3])`?

**Bài 2 (tính tay):** `nums1 = [4, 1, 3]`, `nums2 = [2, 5, 1]`: liệt kê 6 hoán vị và tổng tích, xác nhận đáp án tham lam.

**Bài 3 (điền TODO):** hoàn thiện `max_minus_min(nums1, nums2)` trả về hiệu giữa tổng tích lớn nhất và nhỏ nhất.

**Bài 4 (tự viết lại):** viết lại Cách 3 sao cho trả về luôn **mảng `nums1` đã sắp lại** (độ dài `n`) đạt tối ưu, vẫn `O(n + V)`.

**Bài 5 (mini challenge):** chọn **đúng `k`** phần tử từ `nums1` và `k` phần tử từ `nums2` (số dương), ghép thành `k` cặp, tối thiểu tổng tích. Viết vét cạn cho `n ≤ 6` và thử giả thuyết "lấy `k` nhỏ nhất của mỗi mảng rồi ghép ngược chiều". Giả thuyết có đúng không? (Biến thể tự đặt.)

In [25]:
def max_minus_min(nums1, nums2):
    # TODO: sort cùng chiều cho max, ngược chiều cho min
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[1,2]` tăng, `[4,3]` giảm. (2) `1·4 + 2·3 = 10`. (3) `[3,3,3]`: mọi cách đều `3·(1+2+3) = 18`.

**Bài 2:** (1) Sắp `nums1 = [1,3,4]`, `nums2` giảm `[5,2,1]`. (2) `5 + 6 + 4 = 15`. (3) Liệt kê bằng `permutations` để xác nhận 15 là min.

**Bài 3:** (1) `max = product_sum(sorted(a), sorted(b))`. (2) `min` như Cách 2. (3) Hiệu luôn `≥ 0`.

**Bài 4:** (1) Cần biết **chỉ số** của từng phần tử `nums2` theo giá trị giảm. (2) Gom chỉ số theo giá trị: `pos[v]` = danh sách chỉ số có `nums2 = v`. (3) Duyệt `v` từ 100 xuống, lấy giá trị nhỏ nhất còn lại của `nums1` (con trỏ trên bảng đếm `c1`).

**Bài 5:** (1) Vét cạn: chọn tập con kích thước `k` mỗi bên, rồi min theo hoán vị. (2) Với số dương, tổng tích nhỏ nhất khi chọn các số nhỏ. (3) Kiểm chứng giả thuyết bằng máy.

## 19. Đáp án tham khảo

In [26]:
from itertools import combinations

# Bài 1
assert Solution().minProductSum([1, 2], [3, 4]) == 10 and Solution().minProductSum([3, 3, 3], [1, 2, 3]) == 18

# Bài 2
sums = {p: product_sum(p, [2, 5, 1]) for p in permutations([4, 1, 3])}
for p, v in sorted(sums.items(), key=lambda kv: kv[1]):
    print(p, v)
assert min(sums.values()) == 15 == Solution().minProductSum([4, 1, 3], [2, 5, 1])

# Bài 3
def max_minus_min_ans(a, b):
    return product_sum(sorted(a), sorted(b)) - product_sum(sorted(a), sorted(b, reverse=True))
assert max_minus_min_ans([1, 2], [3, 4]) == 1

# Bài 4
def arrangement_counting(nums1, nums2, V=100):
    c1 = [0] * (V + 1)
    for x in nums1: c1[x] += 1
    pos = [[] for _ in range(V + 1)]
    for idx, y in enumerate(nums2): pos[y].append(idx)
    res = [0] * len(nums1); i = 1
    for v in range(V, 0, -1):
        for idx in pos[v]:
            while c1[i] == 0: i += 1
            res[idx] = i; c1[i] -= 1
    return res

random.seed(101)
for _ in range(500):
    n = random.randint(1, 30)
    a = [random.randint(1, 100) for _ in range(n)]; b = [random.randint(1, 100) for _ in range(n)]
    arr = arrangement_counting(a, b)
    assert sorted(arr) == sorted(a) and product_sum(arr, b) == min_product_sort(a, b)
print("Bài 1-4: OK")

(3, 1, 4) 15
(4, 1, 3) 16
(1, 3, 4) 21
(4, 3, 1) 24
(1, 4, 3) 25
(3, 4, 1) 27
Bài 1-4: OK


In [27]:
# Bài 5 — chọn k cặp
def choose_k_brute(a, b, k):
    best = None
    for A in combinations(a, k):
        for B in combinations(b, k):
            v = product_sum(sorted(A), sorted(B, reverse=True))
            best = v if best is None else min(best, v)
    return best

def choose_k_guess(a, b, k):
    A = sorted(a)[:k]; B = sorted(b)[:k]
    return product_sum(A, sorted(B, reverse=True))

random.seed(102)
counter = None
for _ in range(3000):
    n = random.randint(2, 6); k = random.randint(1, n)
    a = [random.randint(1, 9) for _ in range(n)]; b = [random.randint(1, 9) for _ in range(n)]
    if choose_k_guess(a, b, k) != choose_k_brute(a, b, k):
        counter = (a, b, k); break
print("phản ví dụ cho giả thuyết:", counter)
if counter:
    a, b, k = counter
    print("giả thuyết =", choose_k_guess(a, b, k), "| tối ưu =", choose_k_brute(a, b, k))

phản ví dụ cho giả thuyết: None


Kết luận Bài 5 (theo output: không có phản ví dụ trên 3000 bộ): với **số dương**, giả thuyết **đúng**. Chứng minh ngắn: giả sử tập được chọn của `nums1` chứa `x` nhưng bỏ sót `x' < x`. Thay `x` bằng `x'` (giữ nguyên cặp ghép) làm tổng giảm `(x − x') · y > 0` vì `y > 0` — mâu thuẫn với tối ưu. Vậy tập tối ưu là `k` phần tử nhỏ nhất ở mỗi bên, sau đó ghép ngược chiều.

Điều kiện "số dương" là then chốt: với số âm, lấy số **lớn** (dương) ở một bên ghép với số âm ở bên kia có thể làm tổng nhỏ hơn. Cell dưới kiểm chứng giả thuyết **sai** khi cho phép số âm.

In [28]:
random.seed(103)
found = None
for _ in range(5000):
    n = random.randint(2, 5); k = random.randint(1, n)
    a = [random.randint(-5, 5) for _ in range(n)]; b = [random.randint(-5, 5) for _ in range(n)]
    if choose_k_guess(a, b, k) != choose_k_brute(a, b, k):
        found = (a, b, k); break
a, b, k = found
print(f"phản ví dụ có số âm: nums1={a}, nums2={b}, k={k} | giả thuyết = {choose_k_guess(a, b, k)}, "
      f"tối ưu = {choose_k_brute(a, b, k)}")

phản ví dụ có số âm: nums1=[-4, 4, -4, 5, 1], nums2=[2, 0, 5, -4, 2], k=2 | giả thuyết = 16, tối ưu = -40


## 20. Template nhớ nhanh + Checklist

```python
# Min tổng tích khi được sắp lại: một mảng tăng, một mảng giảm
return sum(x * y for x, y in zip(sorted(nums1), sorted(nums2, reverse=True)))

# Giá trị nhỏ (<= V): counting + hai con trỏ (i tăng trên nums1, j giảm trên nums2)
```

**Checklist trước khi submit:**
- [ ] Ghép **ngược chiều** (nhỏ × lớn) cho tổng nhỏ nhất.
- [ ] Sắp **cả hai** mảng (hoặc dùng bảng đếm cho cả hai).
- [ ] Counting sort: trừ số lượng ở cả hai bảng.
- [ ] Không sửa input nếu không cần (dùng `sorted` thay `sort`).
- [ ] C++/Java: tổng tới `10⁹` — dùng `long`.